In [0]:
alpha_key = "R7MISY8AY7O0KCXM"

In [0]:
print("Alpha key loaded:", alpha_key[:4] + "****")

Alpha key loaded: R7MI****


In [0]:
%pip install kafka-python
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/309.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 307.2/309.8 kB 11.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.8/309.8 kB 5.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install azure-eventhub
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/317.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 174.1/317.1 kB 5.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import time
import random
from datetime import datetime, timezone

import requests
from azure.eventhub import EventData, EventHubProducerClient

# ------------------------------------------------------------------
# YOUR real Event Hub connection string (the one you said worked)
# ------------------------------------------------------------------
EVENT_HUB_CONN_STR = (
    "Endpoint=sb://crypto-pipeline.servicebus.windows.net/;"
    "SharedAccessKeyName=send-policy;"
    "SharedAccessKey=xTiwXp9nlb34L4PyrxAbf0dqZ6/LsRRlr+AEhC5Rjzc=;"
    "EntityPath=crypto-stream"
)
# note: since EntityPath is already in the string, we don't pass eventhub_name

ALPHA_VANTAGE_API_KEY = "YOUR_ALPHA_VANTAGE_KEY_HERE"

SYMBOLS = ["BTC", "ETH", "SOL", "XRP", "LTC"]

last_prices = {
    "BTC": 106000.0,
    "ETH": 3600.0,
    "SOL": 165.0,
    "XRP": 2.4,
    "LTC": 108.0,
}

def fetch_price(symbol: str):
    url = (
        "https://www.alphavantage.co/query"
        f"?function=CURRENCY_EXCHANGE_RATE&from_currency={symbol}&to_currency=USD&apikey={ALPHA_VANTAGE_API_KEY}"
    )
    try:
        resp = requests.get(url, timeout=5)
        data = resp.json()

        if "Information" in data:
            print(f"⚠️ Alpha Vantage rate-limit for {symbol}: {data['Information']}")
            return None

        info = data.get("Realtime Currency Exchange Rate")
        if not info:
            print(f"⚠️ Unexpected payload for {symbol}: {data}")
            return None

        price_str = info.get("5. Exchange Rate")
        if not price_str:
            print(f"⚠️ No price for {symbol}: {data}")
            return None

        return float(price_str)
    except Exception as e:
        print(f"⚠️ fetch_price error for {symbol}: {e}")
        return None

print("🚀 Starting Event Hubs producer (real API + fallback) ...")

# 👇 no eventhub_name here because EntityPath is already in the conn string
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONN_STR
)

while True:
    batch = producer.create_batch()

    for sym in SYMBOLS:
        real_price = fetch_price(sym)

        if real_price is not None:
            price = real_price
            is_simulated = False
            source = "alpha_vantage"
        else:
            base = last_prices.get(sym, 100.0)
            price = round(base * (1 + random.uniform(-0.002, 0.002)), 4)
            is_simulated = True
            source = "alpha_vantage_fallback"

        last_prices[sym] = price

        msg = {
            "symbol": sym,
            "price": price,
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "source": source,
            "is_simulated": is_simulated,
        }

        batch.add(EventData(json.dumps(msg)))
        print("✅ queued:", msg)

    producer.send_batch(batch)
    print("📤 sent batch to Event Hub (crypto-stream)")
    time.sleep(30)

🚀 Starting Event Hubs producer (real API + fallback) ...
⚠️ Alpha Vantage rate-limit for BTC: We have detected your API key as YOUR_ALPHA_VANTAGE_KEY_HERE and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.
✅ queued: {'symbol': 'BTC', 'price': 106023.4389, 'ts_utc': '2025-11-10T22:39:42.566753+00:00', 'source': 'alpha_vantage_fallback', 'is_simulated': True}
⚠️ Alpha Vantage rate-limit for ETH: We have detected your API key as YOUR_ALPHA_VANTAGE_KEY_HERE and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.
✅ queued: {'symbol': 'ETH', 'price': 3603.7259, 'ts_utc': '2025-11-10T22:39:42.647101+00:00', 'source': 'alpha_vantage_fallback', 'is_simulated': True}
⚠️ Alpha Vantage rate-limit for SOL: We have detected your API key as YOU